# Nurse Navigation — Data QA & Assumption Validation

Before trusting any downstream number, this notebook stress-tests the assumptions the analysis is built on. Each section prints raw evidence so the classification logic can be confirmed or corrected against what is actually in the data.

Run top to bottom. Where a check flags a problem, the notebook says so in plain language. Share the outputs and the flagged items can be reconciled before the analysis is quoted.

## 1. Setup and load

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 200)

DATA_DIR = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
SOURCE_FILE = "data_april2026-aug2026.xlsx"

def clean_col(c): return re.sub(r"_+","_",re.sub(r"[^\w]+","_",str(c).strip())).lower()
def read_any(p, sheet=0):
    low=p.lower()
    if low.endswith((".xlsx",".xls")): return pd.read_excel(p, sheet_name=sheet)
    if low.endswith(".parquet"): return pd.read_parquet(p)
    return pd.read_csv(p, low_memory=False)
raw = read_any(os.path.join(DATA_DIR, SOURCE_FILE)); raw.columns=[clean_col(c) for c in raw.columns]
print(f"{len(raw):,} rows, {raw.shape[1]} columns")
FLAGS = []
def flag(ok, msg):
    tag = "PASS" if ok else "CHECK"
    FLAGS.append((tag, msg)); print(f"[{tag}] {msg}")

## 2. Column resolution — did every field map to the right column?

If any field resolves to None or to the wrong column, everything downstream is wrong. Confirm each resolved column name looks right.

In [ ]:
def find_col(df, exact, contains=None):
    norm=lambda x:x.strip("_"); nrm={norm(c):c for c in df.columns}
    for c in exact:
        if c in df.columns: return c
        if norm(c) in nrm: return nrm[norm(c)]
    for pat in (contains or []):
        hits=[c for c in df.columns if pat in c]
        if hits: return sorted(hits,key=len)[0]
    return None
NOTES  = find_col(raw, ["nurses_notes","nurse_notes","notes"], ["nurses_note","note"])
DATE   = find_col(raw, ["transaction_create_date_time_eastern"], ["date_time"])
NMTARA = find_col(raw, ["transaction_breakout_including_bls_nmtara_breakout"], ["nmtara","breakout"])
DISPO  = find_col(raw, ["transaction_response_names","response_macro","response"], ["response_name"])
MARKET = find_col(raw, ["market_name","market"], ["market"])
CAUSE  = find_col(raw, ["cause","chief_complaint"], ["cause","complaint"])
resolved = {"notes":NOTES,"date":DATE,"nmtara":NMTARA,"dispo":DISPO,"market":MARKET,"cause":CAUSE}
display(pd.DataFrame({"field":list(resolved),"resolved_column":list(resolved.values())}))
for f,c in resolved.items():
    flag(c is not None, f"{f} resolved to '{c}'" if c else f"{f} did NOT resolve - fix find_col")

## 3. Disposition codes — the full inventory

Every distinct disposition value with its count. This is the ground truth the keyword flags depend on. Read this table and confirm which codes mean ambulance, self-care, urgent, virtual, referral, other.

In [ ]:
disp = raw[DISPO].fillna("(null)").astype(str).str.strip()
disp_counts = disp.value_counts(dropna=False).rename("calls").to_frame()
disp_counts["pct"] = (disp_counts["calls"]/len(raw)*100).round(2)
display(disp_counts)
print(f"{disp.nunique()} distinct disposition codes")

### 3a. How the current keyword flags classify each code

Shows exactly which codes the current `ambulance|bls|als|911` / `self` / `urgent` logic catches, so miscategorized codes are visible. Watch specifically for codes that should be ambulance but are not caught (for example NN ER).

In [ ]:
_d = disp.str.lower()
audit = disp_counts.copy()
audit["kw_is_ambulance"] = disp_counts.index.to_series().str.lower().str.contains("ambulance|bls|als|911", regex=True)
audit["kw_is_self_care"] = disp_counts.index.to_series().str.lower().str.contains("self")
audit["kw_is_urgent"]    = disp_counts.index.to_series().str.lower().str.contains("urgent")
audit["kw_is_virtual"]   = disp_counts.index.to_series().str.lower().str.contains("virtual|telehealth|video", regex=True)
audit["kw_class"] = np.select(
    [audit["kw_is_ambulance"], audit["kw_is_self_care"], audit["kw_is_urgent"], audit["kw_is_virtual"]],
    ["ambulance","self_care","urgent","virtual"], default="OTHER/UNMATCHED")
display(audit[["calls","pct","kw_class"]])
unmatched = audit[audit["kw_class"]=="OTHER/UNMATCHED"]
flag(len(unmatched)==0, f"{len(unmatched)} disposition codes fall through to OTHER ({unmatched['calls'].sum():,} calls) - confirm these are intentionally 'other'")
er_like = audit[(audit.index.to_series().str.contains(r"\bER\b|emergency", case=False, regex=True)) & (~audit["kw_is_ambulance"])]
flag(len(er_like)==0, f"{len(er_like)} ER-like codes are NOT flagged as ambulance ({er_like['calls'].sum():,} calls) - e.g. NN ER; decide if these are ambulance/ED dispatches")
if len(er_like): display(er_like[["calls","pct","kw_class"]])

## 4. NMTARA level parsing — does the breakout text decode correctly?

The single most load-bearing parse in the analysis. Confirms the level extractor produces sensible 0-6 values and is not silently returning blanks or wrong digits.

In [ ]:
def nmtara_level(x):
    t=str(x)
    m=re.search(r"(?i)n[am]?tara[^0-9]{0,6}(\d)",t)
    if m: return int(m.group(1))
    if re.search(r"(?i)self[- ]?care",t): return np.nan
    m=re.search(r"(?<![0-9])([0-6])(?![0-9])",t)
    return int(m.group(1)) if m else np.nan
lvl = raw[NMTARA].apply(nmtara_level)
dist = lvl.value_counts(dropna=False).sort_index().rename("calls").to_frame()
dist["pct"]=(dist["calls"]/len(raw)*100).round(2)
display(dist)
na_share = lvl.isna().mean()*100
flag(na_share < 40, f"{na_share:.1f}% of rows have NO parsed NMTARA level (NaN) - high values mean the parser is missing the level")
flag((lvl.isin([1,2]).sum())>0, f"high-acuity (NMTARA 1-2) rows found: {lvl.isin([1,2]).sum():,} - if 0, the parser is not catching levels 1-2")

### 4a. Raw breakout text vs parsed level — spot check

Samples the actual breakout strings next to what the parser produced, so a misparse is visible by eye. Look for any row where the parsed level does not match the text.

In [ ]:
chk = raw[[NMTARA]].copy(); chk["parsed_level"]=lvl
sample_by_text = (chk.assign(_t=chk[NMTARA].astype(str))
                  .drop_duplicates("_t").head(40)[[NMTARA,"parsed_level"]])
display(sample_by_text)

### 4b. Distinct breakout strings the parser could not read

Every breakout value that produced a blank level, with counts. If a real acuity level is hiding in these, the parser regex needs to be widened.

In [ ]:
unread = chk[chk["parsed_level"].isna()][NMTARA].astype(str).value_counts().head(30).rename("calls").to_frame()
display(unread)
flag(unread["calls"].sum() < len(raw)*0.4, f"{unread['calls'].sum():,} rows have unparseable breakout text - inspect the strings above")

## 5. Cross-check: NMTARA level vs disposition

Sanity test the two fields against each other. High-acuity levels (1-2) should mostly carry ambulance-type dispositions; self-care disposition should mostly sit at low acuity or self-care. Large surprises here mean one of the two fields is mis-derived.

In [ ]:
tmp = raw.copy(); tmp["lvl"]=lvl; tmp["disp"]=disp
ct = pd.crosstab(tmp["lvl"].fillna("NaN"), tmp["disp"], margins=True)
display(ct)

### 5a. Where do high-acuity (NMTARA 1-2) calls actually go?

The heart of the leakage question. Shows the disposition mix for NMTARA 1-2 calls directly from the codes, before any keyword flag is applied - so the leak number can be judged against raw dispositions, not a derived flag.

In [ ]:
ha_raw = tmp[tmp["lvl"].isin([1,2])]
print(f"NMTARA 1-2 calls: {len(ha_raw):,} ({len(ha_raw)/len(raw)*100:.1f}% of all)")
if len(ha_raw):
    ha_disp = ha_raw["disp"].value_counts().rename("calls").to_frame()
    ha_disp["pct_of_high_acuity"]=(ha_disp["calls"]/len(ha_raw)*100).round(1)
    display(ha_disp)
else:
    print("No NMTARA 1-2 calls parsed - the leak analysis has an empty numerator. Fix the parser (Section 4) first.")

## 6. Override definition check

An override is triage 1-5 with an ambulance disposition. Confirms the two ingredients exist and that the count is not accidentally zero or the whole file.

In [ ]:
amb_kw = disp.str.lower().str.contains("ambulance|bls|als|911", regex=True)
override = tmp["lvl"].between(1,5) & amb_kw.values
print(f"Override (kw ambulance def): {override.sum():,} ({override.mean()*100:.1f}%)")
amb_incl_er = amb_kw | disp.str.contains(r"\bER\b|emergency", case=False, regex=True)
override2 = tmp["lvl"].between(1,5) & amb_incl_er.values
print(f"Override (ambulance INCLUDING NN ER/ED): {override2.sum():,} ({override2.mean()*100:.1f}%)")
flag(abs(override.sum()-override2.sum())/max(override.sum(),1) < 0.2,
     f"Override count changes by {(override2.sum()-override.sum()):,} calls when NN ER counts as ambulance - resolve the NN ER question")

## 7. Notes coverage — is there enough text to read?

The LLM half depends on real note text. Quantifies truly missing notes and notes too short to be useful.

In [ ]:
notes = raw[NOTES].astype(str)
n_null = raw[NOTES].isna().sum()
n_blank = (notes.str.strip().isin(["","nan","none","n/a","null"])).sum()
n_short = (notes.str.strip().str.len() < 20).sum()
cov = pd.DataFrame({"metric":["true null","blank/na-like","under 20 chars (skipped by LLM)","usable (>=20 chars)"],
                    "calls":[int(n_null), int(n_blank), int(n_short), int((notes.str.strip().str.len()>=20).sum())]})
cov["pct"]=(cov["calls"]/len(raw)*100).round(1)
display(cov)
flag(n_short/len(raw) < 0.5, f"{n_short/len(raw)*100:.1f}% of notes are too short for the LLM - high values shrink the effective sample")

## 8. Duplicates and identity

Checks whether one physical call appears as multiple rows (which would inflate every count). Uses the work-set and external-reference IDs if present.

In [ ]:
WORKSET = find_col(raw, ["work_set_id"], ["work_set","workset"])
EPISODE = find_col(raw, ["external_reference_number","incident_id"], ["external_reference","incident"])
print("full-row duplicates:", raw.duplicated().sum())
for idc in [WORKSET, EPISODE]:
    if idc:
        d = raw[idc].dropna()
        dup = d.duplicated().sum()
        print(f"{idc}: {d.nunique():,} unique / {len(d):,} non-null  ({dup:,} repeat rows)")
        flag(dup/max(len(d),1) < 0.1, f"{idc} repeats on {dup:,} rows - confirm whether a call legitimately spans multiple rows")

## 9. Date coverage and gaps

Confirms the date field parses and the timeline has no unexpected holes that would distort the trend charts.

In [ ]:
dt = pd.to_datetime(raw[DATE], errors="coerce")
bad = dt.isna().sum()
print(f"unparseable dates: {bad:,} ({bad/len(raw)*100:.1f}%)")
by_month = dt.dropna().dt.to_period("M").value_counts().sort_index()
display(by_month.rename("calls").to_frame())
flag(bad/len(raw) < 0.05, f"{bad/len(raw)*100:.1f}% of dates did not parse")
flag(by_month.min() > by_month.max()*0.1, "one or more months have unusually low volume - check for a partial-month or data gap")

## 10. Market field sanity

Confirms markets are populated and not dominated by a null or catch-all value that would skew per-market analysis.

In [ ]:
mk = raw[MARKET].fillna("(null)").astype(str)
mkc = mk.value_counts().rename("calls").to_frame(); mkc["pct"]=(mkc["calls"]/len(raw)*100).round(1)
display(mkc.head(25))
print(f"{mk.nunique()} distinct markets")
flag(mk.eq("(null)").mean() < 0.05, f"{mk.eq('(null)').mean()*100:.1f}% of rows have no market")

## 11. QA summary

Every check in one place. CHECK rows are the ones to resolve before quoting the analysis.

In [ ]:
summary = pd.DataFrame(FLAGS, columns=["status","check"])
display(summary)
n_check = (summary["status"]=="CHECK").sum()
print(f"\n{n_check} item(s) need review, {(summary['status']=='PASS').sum()} passed.")
if n_check: print("Share the CHECK rows and the tables above to reconcile before trusting downstream numbers.")